# Scaling Law Analysis: Make Better Training Decisions

This is the core hands-on notebook for the A5 Scaling Laws session. No GPU, no network access, and no special setup beyond a local Python environment are required — everything here runs on the 24 real training runs bundled in `data/results.csv`.

**What we'll do:**
1. Load → Check → Select → Plot the raw sweep data
2. Fit a descriptive scaling law across all four compute budgets
3. Hold out the largest budget and check whether the fit predicts it
4. Turn the fit into a concrete recommendation: what should we train next?
5. Exercise: reproduce the workflow on a different holdout (Path A), then adapt it to new data (Path B)

If you have Aurora access, `02_live_training_bonus.ipynb` lets you submit one real training run and fold it into this analysis — but it's entirely optional and this notebook does not depend on it.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scaling_lib import core

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load → Check → Select → Plot

**Load:** read the sweep results. Each row is one completed training run at a given model depth and FLOPs budget.

**Check — what must stay comparable before we trust a comparison across runs:**
- Same metric (`val_bpb`, bits-per-byte on a held-out validation split) and the same evaluation procedure for every run
- Same held-out evaluation data, with no leakage into training
- Same data distribution (or a clearly documented change)
- A consistent architecture-scaling rule and a fair, documented tuning policy across sizes (not necessarily identical hyperparameters — different sizes may need different learning rates)
- A defined compute accounting convention and the *actual* achieved budget per run (not just the target)
- Consistent treatment of failed runs and random seeds

All 24 runs here were produced by the same training harness, same tokenizer/vocab, same eval split — so we can compare them directly.

In [ ]:
df = core.load_raw()
print(f"{len(df)} runs across {df['flops_budget'].nunique()} FLOPs budgets")
print("Budgets:", sorted(df["flops_budget"].unique()))
df.head()

In [ ]:
# Select: the best (lowest val_bpb) run at each budget
winners = core.best_per_budget(df)
winners[["flops_budget", "depth", "effective_params", "tokens_trained", "val_bpb"]]

In [ ]:
# Plot: every run, with the per-budget winner highlighted
fig, ax = plt.subplots(figsize=(7, 5))
for budget, group in df.groupby("flops_budget"):
    ax.scatter(group["effective_params"], group["val_bpb"], label=f"{budget:.0e}", alpha=0.5)
ax.scatter(winners["effective_params"], winners["val_bpb"], color="black", marker="*", s=200,
           label="best per budget", zorder=5)
ax.set_xscale("log")
ax.set_xlabel("Effective parameters (N)")
ax.set_ylabel("Validation bits-per-byte")
ax.set_title("All runs by FLOPs budget")
ax.legend(title="FLOPs budget")
plt.show()

## 2. Four-budget descriptive fit

We fit a straight line to log loss versus log compute, using the best-observed point at each budget. This is a **descriptive fit**: it summarizes the trend we measured, not a guaranteed law that holds outside the range we tested.

In [ ]:
slope, coeff = core.fit_power_law(winners["flops_budget"].values, winners["val_bpb"].values)
print(f"Four-budget descriptive fit: val_bpb = {coeff:.4f} * C^{slope:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(winners["flops_budget"], winners["val_bpb"], color="black", s=80, zorder=5, label="best per budget")
c_range = np.logspace(np.log10(winners["flops_budget"].min()), np.log10(winners["flops_budget"].max()), 50)
ax.plot(c_range, coeff * c_range ** slope, "--", label="four-budget descriptive fit")
ax.set_xscale("log")
ax.set_xlabel("FLOPs budget (C)")
ax.set_ylabel("Best validation bits-per-byte")
ax.legend()
plt.show()

## 3. Check the forecast on a held-out compute budget

This is a **retrospective budget holdout**, not an untouched test set: the same validation metric picked each budget's winner, which can introduce selection optimism when measurements are noisy. Still, it's a useful sanity check — we refit on the three smaller budgets only, then see whether that fit predicts the budget we withheld.

**Acceptance criterion:** is the prediction error small relative to the improvement we expect — and would it change the configuration we choose?

In [ ]:
held_out_budget = winners["flops_budget"].max()  # largest budget, matches the deck's narrative
train_mask = winners["flops_budget"] != held_out_budget

train_winners = winners[train_mask]
held_out_row = winners[~train_mask].iloc[0]

slope3, coeff3 = core.fit_power_law(train_winners["flops_budget"].values, train_winners["val_bpb"].values)
print(f"Three-budget prediction fit: val_bpb = {coeff3:.4f} * C^{slope3:.4f}")

predicted_bpb = coeff3 * held_out_row["flops_budget"] ** slope3
actual_bpb = held_out_row["val_bpb"]
# Use unrounded values for the relative error, not the printed/rounded numbers above
relative_error = abs(predicted_bpb - actual_bpb) / actual_bpb

print(f"\nHeld-out budget: {held_out_budget:.0e}")
print(f"Predicted val_bpb: {predicted_bpb:.6f}")
print(f"Actual val_bpb:    {actual_bpb:.6f}")
print(f"Relative error:    {relative_error:.2%}")

## 4. What should we train at the next budget?

A loss forecast answers "what loss should we expect?" but not "what model size and token count should we actually use?" The table below reports each budget's best-observed configuration and whether that winner sits at the **edge** of the sizes we tested — a winner at the edge means we haven't actually found the optimum, only the best of a too-narrow sweep.

In [ ]:
alloc = core.allocation_table(df)
alloc

In [ ]:
edge_rows = alloc[alloc["range_position"].str.startswith("max") | alloc["range_position"].str.startswith("min")]
for _, row in edge_rows.iterrows():
    print(f"Budget {row['flops_budget']:.0e}: winner is at the {row['range_position']} tested sizes — "
          f"widen the sweep here before trusting a size-vs-budget trend.")

print("\nRecommendation: at the next budget, test larger neighboring model sizes and allocate "
      "training tokens via D ≈ C / (6N). Any size-vs-budget trend from only four budgets is "
      "tentative, not a validated optimum.")

## 5. Exercise

Pick a path (or do both if you have time):

- **Path A — Reproduce:** repeat the load → select → fit → holdout workflow, but hold out a *different* budget than the one we used above. Does the fit still predict it well? Does it predict it worse?
- **Path B — Adapt:** apply the same `scaling_lib.core` functions to a different dataset — a toy example is provided, or swap in metrics from your own project (any CSV with budget/size/metric columns works, as long as you rename columns to match).

### Path A: reproduce with a different holdout

Fill in the blanks (marked `___`) below, then run the solution cell to check your answer.

Try it yourself in a new cell before checking the solution below:

```python
# Try holding out the smallest budget instead of the largest.
my_held_out_budget = ___  # hint: winners["flops_budget"].min()
my_train_mask = winners["flops_budget"] != my_held_out_budget

my_train_winners = winners[___]
my_held_out_row = winners[~my_train_mask].iloc[0]

my_slope, my_coeff = core.fit_power_law(___, ___)
my_predicted = my_coeff * my_held_out_row["flops_budget"] ** my_slope
my_actual = my_held_out_row["val_bpb"]
my_relative_error = abs(my_predicted - my_actual) / my_actual
print(f"Relative error holding out {my_held_out_budget:.0e}: {my_relative_error:.2%}")
```

In [ ]:
# SOLUTION — run this cell to check your answer, or if you'd rather skip ahead
my_held_out_budget = winners["flops_budget"].min()
my_train_mask = winners["flops_budget"] != my_held_out_budget

my_train_winners = winners[my_train_mask]
my_held_out_row = winners[~my_train_mask].iloc[0]

my_slope, my_coeff = core.fit_power_law(my_train_winners["flops_budget"].values, my_train_winners["val_bpb"].values)
my_predicted = my_coeff * my_held_out_row["flops_budget"] ** my_slope
my_actual = my_held_out_row["val_bpb"]
my_relative_error = abs(my_predicted - my_actual) / my_actual
print(f"Relative error holding out {my_held_out_budget:.0e}: {my_relative_error:.2%}")
print("\nExtrapolating backward (predicting the smallest budget from the three largest) is a "
      "different, often harder, test than extrapolating forward — compare the two error rates.")

### Path B: adapt the workflow to new data

The functions in `scaling_lib.core` only assume a dataframe with `flops_budget`, `depth`, `model_dim`, `params_transformer`, `tokens_trained`, and `val_bpb` columns — they don't know anything about nanochat specifically. If you have results from your own project (a sweep over model size and a resource budget, with some quality metric), rename your columns to match and the same `load_raw` / `best_per_budget` / `fit_power_law` / `allocation_table` calls apply directly.

A toy synthetic dataset is provided below so you can see the workflow generalize even without your own data on hand.

In [ ]:
# A synthetic toy sweep with the same column names, standing in for "your own project's data."
rng = np.random.default_rng(0)
toy_rows = []
for budget in [5e16, 1e17, 2e17]:
    for depth in [6, 8, 10, 12]:
        model_dim = depth * 64
        params_transformer = depth * model_dim * model_dim * 12
        tokens_trained = int(budget / (6 * (params_transformer + model_dim * core.VOCAB_SIZE)))
        noise = rng.normal(0, 0.01)
        val_bpb = 1.5 * (budget ** -0.05) + 0.01 * abs(depth - 9) + noise
        toy_rows.append({
            "flops_budget": budget, "depth": depth, "model_dim": model_dim,
            "params_transformer": params_transformer, "tokens_trained": tokens_trained,
            "val_bpb": val_bpb,
        })
toy_df = core.compute_effective_params(pd.DataFrame(toy_rows))

toy_winners = core.best_per_budget(toy_df)
toy_slope, toy_coeff = core.fit_power_law(toy_winners["flops_budget"].values, toy_winners["val_bpb"].values)
print(f"Toy-data fit: val_bpb = {toy_coeff:.4f} * C^{toy_slope:.4f}")
core.allocation_table(toy_df)

## Summary

| Question | Method |
|---|---|
| What loss should we expect at a larger budget? | Four-budget descriptive power-law fit, checked against a held-out budget |
| What should we actually train next? | Per-budget allocation table, flagging winners at the edge of the tested size range |
| Does this generalize beyond this dataset? | Same functions, any CSV with the right columns (Path B) |

**Caveats to carry forward:** this is a descriptive fit from a small number of budgets, with no run-to-run variance estimate and no loss-floor term — treat any extrapolation far outside the tested range with real skepticism, and prefer widening the sweep over trusting a single fitted curve.

If you have Aurora access, continue to `02_live_training_bonus.ipynb` to submit one real training run and see how it folds into this analysis.